# 04 - Tabelas gerenciadas × não gerenciadas

Item 4 do trabalho: comparar o comportamento das tabelas nos dois laboratórios.

| | **Gerenciada (interna)** | **Não gerenciada (externa)** |
|---|---|---|
| Quem cuida dos dados | o Spark (dados **e** metadados) | o usuário; o Spark só guarda os metadados |
| Como é criada | sem `LOCATION`, ex.: `df.write.saveAsTable(...)` ou `CREATE TABLE t (...) USING delta` | com `LOCATION`, apontando para arquivos que já existem |
| Onde ficam os dados | `spark.sql.warehouse.dir` (aqui: `s3a://warehouse/spark-warehouse/`) | onde o `LOCATION` indicar (aqui: `s3a://bronze/<tabela>`) |
| `DROP TABLE` | apaga metadados **e arquivos** | apaga só os metadados; **arquivos preservados** |

Este notebook prova a última linha na prática, usando a tabela `categoria` da bronze.

**Pré-requisito:** notebook `02` executado.

## 1. SparkSession

In [ ]:
import sys
from pathlib import Path

# Garante que o pacote do projeto (src/spark_delta) seja encontrado mesmo que o
# kernel não tenha sido instalado com `uv sync` (fallback para src/).
try:
    import spark_delta  # noqa: F401
except ModuleNotFoundError:
    sys.path.insert(0, str(Path.cwd().resolve().parent / "src"))
    sys.path.insert(0, str(Path.cwd().resolve() / "src"))

from spark_delta.config import settings
from spark_delta.spark_session import get_spark, WAREHOUSE_DIR
from spark_delta.storage import s3_client, ensure_bucket

s3 = s3_client()
for bucket in (settings.bronze_bucket, settings.warehouse_bucket):
    ensure_bucket(s3, bucket)

spark = get_spark("gerenciadas_vs_externas")
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "| warehouse:", WAREHOUSE_DIR)
spark

In [ ]:
from spark_delta.spark_session import WAREHOUSE_PREFIX
from spark_delta.storage import list_objects

BRONZE_CATEGORIA = f"s3a://{settings.bronze_bucket}/categoria"
PREFIXO_GERENCIADA = f"{WAREHOUSE_PREFIX}/categoria_gerenciada/"

def arquivos(bucket, prefixo):
    return [o["Key"] for o in list_objects(s3, bucket, prefixo)]

def mostrar_arquivos():
    g = arquivos(settings.warehouse_bucket, PREFIXO_GERENCIADA)
    e = arquivos(settings.bronze_bucket, "categoria/")
    print(f"GERENCIADA s3://{settings.warehouse_bucket}/{PREFIXO_GERENCIADA}: {len(g)} arquivos")
    print(f"EXTERNA    s3://{settings.bronze_bucket}/categoria/: {len(e)} arquivos")

# Limpeza de execuções anteriores (o catálogo em memória some ao fim da sessão,
# mas os arquivos de uma tabela gerenciada antiga podem ter ficado no bucket)
for chave in arquivos(settings.warehouse_bucket, PREFIXO_GERENCIADA):
    s3.delete_object(Bucket=settings.warehouse_bucket, Key=chave)
spark.sql("DROP TABLE IF EXISTS categoria_gerenciada")
spark.sql("DROP TABLE IF EXISTS categoria_externa")

## 2. Criar as duas tabelas

- **Gerenciada:** a partir de um DataFrame com `saveAsTable` → o Spark escolhe o local (warehouse) e passa a ser dono dos arquivos.
- **Externa:** `CREATE TABLE ... LOCATION` apontando para a tabela Delta que já está na bronze.

In [ ]:
df_categoria = spark.read.format("delta").load(BRONZE_CATEGORIA)

# GERENCIADA (tabela a partir de DataFrame, sem LOCATION)
df_categoria.write.format("delta").saveAsTable("categoria_gerenciada")

# NÃO GERENCIADA / EXTERNA (com LOCATION)
spark.sql(f"CREATE TABLE categoria_externa USING delta LOCATION '{BRONZE_CATEGORIA}'")

spark.sql("SELECT COUNT(*) AS gerenciada FROM categoria_gerenciada").show()
spark.sql("SELECT COUNT(*) AS externa FROM categoria_externa").show()

## 3. O que o catálogo diz sobre cada uma

In [ ]:
for t in spark.catalog.listTables():
    if t.name.startswith("categoria_"):
        print(f"{t.name:<22} tableType = {t.tableType}")

In [ ]:
for tabela in ("categoria_gerenciada", "categoria_externa"):
    print(f"=== DESCRIBE TABLE EXTENDED {tabela} ===")
    (spark.sql(f"DESCRIBE TABLE EXTENDED {tabela}")
        .where("col_name IN ('Type', 'Provider', 'Location')")
        .show(truncate=False))

In [ ]:
mostrar_arquivos()

## 4. `DROP TABLE` nas duas

Aqui está a diferença principal: o mesmo comando tem efeitos diferentes nos arquivos.

In [ ]:
spark.sql("DROP TABLE categoria_gerenciada")
spark.sql("DROP TABLE categoria_externa")

spark.sql("SHOW TABLES LIKE 'categoria_*'").show()
mostrar_arquivos()

**Resultado esperado:** a pasta da gerenciada no bucket `warehouse` ficou com **0 arquivos** (dados apagados junto com a tabela), enquanto a pasta `categoria/` da bronze continua com todos os arquivos Parquet e o `_delta_log`.

## 5. Recuperar a tabela externa

Como os arquivos (incluindo o `_delta_log`) continuam no MinIO, basta registrar a tabela de novo. Os dados **e o histórico de versões** voltam inteiros.
A gerenciada não tem como voltar: os arquivos foram apagados.

In [ ]:
spark.sql(f"CREATE TABLE categoria_externa USING delta LOCATION '{BRONZE_CATEGORIA}'")
spark.sql("SELECT * FROM categoria_externa ORDER BY cd_categoria").show()
spark.sql("DESCRIBE HISTORY categoria_externa").select("version", "timestamp", "operation").show(truncate=False)

## 6. Comparação entre os dois laboratórios

| | **Trabalho 1** (`Pyspark-Jupyter`) | **Trabalho 2** (este repositório) |
|---|---|---|
| Como as tabelas foram criadas | `CREATE TABLE estoque_delta (...) USING delta` (Delta) e `CREATE TABLE local.estoque_iceberg (...) USING iceberg` (Iceberg), **sem `LOCATION`** | `CREATE TABLE <t> USING delta LOCATION 's3a://bronze/<t>'` |
| Tipo | **Gerenciadas**: o Spark/catálogo decidiu o local (`spark-warehouse/` no disco local) | **Não gerenciadas (externas)**: o local é o bucket `bronze` do MinIO |
| Origem dos dados | criados à mão com `INSERT` dentro do Spark | extraídos do SQL Server → CSV → Delta, gravados por caminho antes de existir qualquer tabela no catálogo |
| Efeito de `DROP TABLE` | apaga também os dados (no Delta); a tabela teria que ser recriada e repopulada | só remove o registro do catálogo; os dados continuam no MinIO e a tabela pode ser registrada de novo |
| Armazenamento | disco local | object storage (S3/MinIO), compartilhável por outras ferramentas |

**Ponto para discussão em aula:** este projeto usa o catálogo padrão do Spark **em memória** (sem Hive Metastore). Os metadados de *qualquer* tabela, gerenciada ou não, somem ao final da sessão; por isso todo notebook registra as tabelas de novo.
Nas externas isso não é problema, porque o `_delta_log` dentro do MinIO é a fonte da verdade (schema, versões e arquivos). Já os arquivos de uma gerenciada ficam "órfãos" no warehouse quando a sessão acaba sem `DROP`, sem nenhum catálogo apontando para eles. Em produção, um metastore persistente (Hive Metastore, Unity Catalog, AWS Glue) resolve isso.

In [ ]:
spark.stop()
print("SparkSession finalizada.")